In [0]:
from pyspark.sql.functions import *

df = spark.table("workspace.bronze.bronze_order_items")

# Cleaning the df (Selecting imp. fields only)
cleaned_df = df.select(
    col("order_item_id"),
    col("order_id"),
    col("product_id"),
    col("quantity"),
    col("unit_price"),
    col("line_total"),
    col("_modified"),
    col("ingestion_ts") 
).filter(col("order_item_id").isNotNull())

In [0]:
missing_values = ['\\n', '?', '', 'null']

# Adding is_valid flag
# Instead of deleting records, Adding missing values to them
cleaned_df = (
    cleaned_df
    .withColumn(
        "is_valid",
        when(
            col("order_id").isNull() |
            col("product_id").isNull() |
            lower(trim(col("order_id"))).isin(missing_values) |
            lower(trim(col("product_id"))).isin(missing_values),
            0
        ).otherwise(1)
    )
    .withColumn(
        "order_id",
        when(
            col("order_id").isNull() |
            lower(trim(col("order_id"))).isin(missing_values),
            None
        ).otherwise(col("order_id"))
    )
    .withColumn(
        "product_id",
        when(
            col("product_id").isNull() |
            lower(trim(col("product_id"))).isin(missing_values),
            None
        ).otherwise(col("product_id"))
    )
)

display(cleaned_df)

In [0]:
cleaned_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.silver_order_items")